# Metropolis-Hastings Sampling with minibayes

This notebook demonstrates the minibayes Model API and MetropolisHastings sampler with three examples:

1. **Normal-Normal Model**: Sampling from a posterior with analytical solution
2. **Bayesian Linear Regression**: 1D regression with priors on slope, intercept, and noise
3. **Multi-Chain Sampling**: Running multiple chains with R-hat convergence diagnostics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import minibayes as mb
from minibayes import Model, dist
from minibayes.diagnostics import effective_sample_size, summary, r_hat
from minibayes.results import InferenceResult

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

---
## Example 1: Normal-Normal Posterior

**Setup:**
- Prior: $\mu \sim N(0, \tau^2)$ with $\tau = 1$
- Likelihood: $y_i \sim N(\mu, \sigma^2)$ with $\sigma = 1$ (known)
- Data: $n$ observations

**Analytical posterior:**
$$\mu | y \sim N(\mu_{post}, \sigma^2_{post})$$

where:
- $\mu_{post} = \frac{n \bar{y} / \sigma^2}{n/\sigma^2 + 1/\tau^2}$
- $\sigma^2_{post} = \frac{1}{n/\sigma^2 + 1/\tau^2}$

In [ ]:
# Generate synthetic data
rng = np.random.default_rng(42)

true_mu = 2.5
sigma = 1.0  # known likelihood std
n = 20
data = rng.normal(true_mu, sigma, size=n)

print(f"True mu = {true_mu}")
print(f"Sample mean = {np.mean(data):.3f}")
print(f"n = {n}")

In [ ]:
# Compute analytical posterior
tau = 1.0  # prior std
y_bar = np.mean(data)

posterior_precision = n / sigma**2 + 1 / tau**2
posterior_var = 1 / posterior_precision
posterior_mean = (n * y_bar / sigma**2) / posterior_precision
posterior_std = np.sqrt(posterior_var)

print(f"Analytical posterior: N({posterior_mean:.3f}, {posterior_std:.3f})")

In [ ]:
# Define minibayes model
priors = {"mu": dist.Normal(loc=0, scale=tau)}


def likelihood(params: dict[str, float], obs: np.ndarray) -> float:
    """Normal likelihood with known variance."""
    mu = params["mu"]
    # Use dist.Normal.log_prob to compute likelihood
    likelihood_dist = dist.Normal(loc=mu, scale=sigma)
    return float(np.sum(likelihood_dist.log_prob(obs)))


model = Model(priors=priors, likelihood=likelihood)
print(f"Parameters: {model.param_names}")
print(f"Transforms: {model.transforms}")

In [ ]:
# Run MCMC using mb.sample()
result = mb.sample(
    model,
    data=data,
    num_samples=5000,
    num_warmup=1000,
    sampler="mh",
    sampler_kwargs={"proposal_scale": 0.5},
    seed=42,
    progress=True,
    num_chains=2
)

# Samples are (num_chains, num_samples) - flatten for analysis
samples = result.samples["mu"].flatten()
ess = effective_sample_size(result.samples["mu"])

# Acceptance rate is always array with shape (num_chains,)
for i, rate in enumerate(result.acceptance_rate):
    print(f"Chain {i+1} acceptance rate: {rate:.1%}")
print(f"Effective Sample Size: {ess:.1f}")
print(f"Empirical mean: {np.mean(samples):.3f} (analytical: {posterior_mean:.3f})")
print(f"Empirical std: {np.std(samples):.3f} (analytical: {posterior_std:.3f})")

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trace plot
axes[0].plot(samples, alpha=0.7, lw=0.5)
axes[0].axhline(posterior_mean, color="red", linestyle="--", label="Analytical mean")
axes[0].axhline(true_mu, color="green", linestyle=":", label=f"True mu = {true_mu}")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("mu")
axes[0].set_title("Trace Plot")
axes[0].legend()

# Histogram vs analytical
x = np.linspace(samples.min() - 0.5, samples.max() + 0.5, 200)
analytical_pdf = (
    1 / np.sqrt(2 * np.pi * posterior_var)
) * np.exp(-0.5 * (x - posterior_mean) ** 2 / posterior_var)

axes[1].hist(samples, bins=50, density=True, alpha=0.7, label="MCMC samples")
axes[1].plot(x, analytical_pdf, "r-", lw=2, label="Analytical posterior")
axes[1].axvline(true_mu, color="green", linestyle=":", lw=2, label=f"True mu = {true_mu}")
axes[1].set_xlabel("mu")
axes[1].set_ylabel("Density")
axes[1].set_title("Posterior Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Example 2: Bayesian Linear Regression

**Model:**
$$y = \alpha + \beta x + \epsilon, \quad \epsilon \sim N(0, \sigma^2)$$

**Priors:**
- $\alpha \sim N(0, 10)$ (intercept)
- $\beta \sim N(0, 10)$ (slope)
- $\sigma \sim \text{HalfNormal}(5)$ (noise std)

In [ ]:
# Generate synthetic data
rng = np.random.default_rng(123)

true_alpha = 1.5
true_beta = 2.0
true_sigma = 0.3
n = 10

x = np.linspace(0, 3, n)
y = true_alpha + true_beta * x + rng.normal(0, true_sigma, n)

print(f"True parameters: alpha={true_alpha}, beta={true_beta}, sigma={true_sigma}")
print(f"Data: {n} points")

In [ ]:
# Plot data
plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.7, label="Data")
plt.plot(x, true_alpha + true_beta * x, "g--", lw=2, label="True line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic Linear Regression Data")
plt.legend()
plt.show()

In [ ]:
# Define minibayes model
priors = {
    "alpha": dist.Normal(loc=0, scale=10),
    "beta": dist.Normal(loc=0, scale=10),
    "sigma": dist.HalfNormal(scale=5),
}


def likelihood(params: dict[str, float], obs: tuple) -> float:
    """Normal likelihood for linear regression."""
    x_data, y_data = obs
    alpha = params["alpha"]
    beta = params["beta"]
    sigma = params["sigma"]

    # Predicted values
    predicted = alpha + beta * x_data

    # Use dist.Normal.log_prob to compute likelihood
    likelihood_dist = dist.Normal(loc=predicted, scale=sigma)
    return float(np.sum(likelihood_dist.log_prob(y_data)))


model = Model(priors=priors, likelihood=likelihood)
print(f"Parameters: {model.param_names}")
print(f"Transforms: {model.transforms}")

In [ ]:
# Run MCMC using mb.sample() with per-parameter proposal scales
result = mb.sample(
    model,
    data=(x, y),
    num_samples=5000,
    num_warmup=1000,
    sampler="mh",
    sampler_kwargs={
        "proposal_scale": {
            "alpha": 0.15,
            "beta": 0.08,
            "sigma": 0.1,  # in unconstrained (log) space
        }
    },
    seed=42,
)

# Flatten samples for single-chain use (shape: (1, num_samples) -> (num_samples,))
samples = {name: arr.flatten() for name, arr in result.samples.items()}
print(f"Acceptance rate: {result.acceptance_rate[0]:.1%}")
print(f"Elapsed time: {result.elapsed_time:.2f}s")

In [ ]:
# Summary statistics using diagnostics.summary()
samples_arrays = {name: np.array(samples[name]) for name in samples}
stats = summary(samples_arrays)

print("\nPosterior Summary:")
print("-" * 70)
print(f"{'Parameter':<10} {'True':>8} {'Mean':>8} {'Std':>8} {'ESS':>8} {'95% CI'}")
print("-" * 70)

for name, true_val in [
    ("alpha", true_alpha),
    ("beta", true_beta),
    ("sigma", true_sigma),
]:
    s = stats[name]
    print(
        f"{name:<10} {true_val:>8.3f} {s['mean']:>8.3f} {s['std']:>8.3f} {s['ess']:>8.1f}   [{s['5%']:.3f}, {s['95%']:.3f}]"
    )

### Using InferenceResult

The `InferenceResult` class wraps MCMC samples with metadata and provides convenient methods for summarization, diagnostics, and persistence.

In [ ]:
# The result from mb.sample() is already an InferenceResult
# Use the summary() method
print("InferenceResult.summary():")
result_stats = result.summary()
for param, stats in result_stats.items():
    print(f"\n{param}:")
    for key, val in stats.items():
        print(f"  {key}: {val:.4f}")

In [ ]:
# Save and load results
result.save("linear_regression_results.npz")
print("Saved results to linear_regression_results.npz")

# Load them back
loaded_result = InferenceResult.load("linear_regression_results.npz")
print(f"Loaded: {loaded_result.num_samples} samples, {loaded_result.num_chains} chain(s)")
print(f"Sampler: {loaded_result.sampler}")
print(f"Acceptance rate: {loaded_result.acceptance_rate[0]:.1%}")

# to_dict() for JSON-compatible output
result_dict = result.to_dict()
print(f"\nto_dict() keys: {list(result_dict.keys())}")

## Diagnostics

minibayes provides convergence diagnostics to assess MCMC quality:
- **ESS (Effective Sample Size)**: Accounts for autocorrelation in the chain. Higher is better.
- **R-hat**: Requires multiple chains to detect non-convergence (not shown in this single-chain example).

In [ ]:
# Trace plots and histograms
fig, axes = plt.subplots(3, 2, figsize=(12, 8))

for i, (name, true_val) in enumerate(
    [("alpha", true_alpha), ("beta", true_beta), ("sigma", true_sigma)]
):
    # Trace
    axes[i, 0].plot(samples[name], alpha=0.7, lw=0.5)
    axes[i, 0].axhline(true_val, color="red", linestyle="--", label="True")
    axes[i, 0].set_ylabel(name)
    axes[i, 0].legend(loc="upper right")

    # Histogram
    axes[i, 1].hist(samples[name], bins=50, density=True, alpha=0.7)
    axes[i, 1].axvline(true_val, color="red", linestyle="--", lw=2, label="True")
    axes[i, 1].axvline(
        np.mean(samples[name]), color="blue", linestyle="-", lw=2, label="Mean"
    )
    axes[i, 1].set_xlabel(name)
    axes[i, 1].legend(loc="upper right")

axes[0, 0].set_title("Trace Plots")
axes[0, 1].set_title("Posterior Distributions")
plt.tight_layout()
plt.show()

In [ ]:
# Posterior predictive plot
fig, ax = plt.subplots(figsize=(10, 6))

# Data
ax.scatter(x, y, alpha=0.7, s=50, label="Data", zorder=3)

# True line
x_plot = np.linspace(0, 3, 100)
ax.plot(
    x_plot, true_alpha + true_beta * x_plot, "g--", lw=2, label="True", zorder=2
)

# Posterior samples (thin for clarity)
for i in range(0, len(samples["alpha"]), 50):
    ax.plot(
        x_plot,
        samples["alpha"][i] + samples["beta"][i] * x_plot,
        "b-",
        alpha=0.03,
        zorder=1,
    )

# Posterior mean
ax.plot(
    x_plot,
    np.mean(samples["alpha"]) + np.mean(samples["beta"]) * x_plot,
    "r-",
    lw=2,
    label="Posterior mean",
    zorder=2,
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Bayesian Linear Regression: Posterior Predictive")
ax.legend()
plt.show()

In [ ]:
# Posterior predictive with 90% predictive interval (including extrapolation)
fig, ax = plt.subplots(figsize=(10, 6))

# Data
ax.scatter(x, y, alpha=0.7, s=50, label="Data", zorder=3)

# Extend x_plot beyond data range to show extrapolation uncertainty
x_plot = np.linspace(-5, 9, 100)
ax.plot(
    x_plot, true_alpha + true_beta * x_plot, "g--", lw=2, label="True", zorder=2
)

# Sample from posterior predictive: y ~ Normal(alpha + beta*x, sigma)
rng_pred = np.random.default_rng(42)
y_mean_samples = samples["alpha"][:, np.newaxis] + samples["beta"][:, np.newaxis] * x_plot
# Add observation noise for each sample
y_pred = y_mean_samples + rng_pred.normal(0, 1, y_mean_samples.shape) * samples["sigma"][:, np.newaxis]

# 90% predictive interval (5th to 95th percentile)
y_lower = np.percentile(y_pred, 5, axis=0)
y_upper = np.percentile(y_pred, 95, axis=0)
y_mean = np.mean(y_mean_samples, axis=0)

# Shaded 90% predictive region
ax.fill_between(x_plot, y_lower, y_upper, alpha=0.3, color="blue", label="90% Predictive Interval")

# Posterior mean
ax.plot(x_plot, y_mean, "r-", lw=2, label="Posterior mean", zorder=2)


ax.set_xlim(-5, 9)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Bayesian Linear Regression: 90% Predictive Interval (with extrapolation)")
ax.legend()
plt.show()

In [ ]:
# Pair plot showing joint distributions
import pandas as pd

df = pd.DataFrame(samples)
g = sns.pairplot(df, diag_kind="kde", plot_kws={"alpha": 0.3, "s": 5})
g.figure.suptitle("Posterior Joint Distribution", y=1.02)
plt.show()

---
## Example 3: Multi-Chain Sampling and R-hat Diagnostic

Running multiple independent chains allows us to:
1. **Assess convergence** using the R-hat (Gelman-Rubin) diagnostic
2. **Better explore the posterior** by starting from different initial points
3. **Detect non-convergence** when chains don't mix well

**R-hat interpretation:**
- R-hat ≈ 1.0: Chains have converged to the same distribution
- R-hat > 1.01: Potential convergence issues, consider running longer

In [ ]:
# Run 4 independent chains using mb.sample()
multi_result = mb.sample(
    model,
    data=(x, y),
    num_samples=5000,
    num_warmup=1500,
    num_chains=4,
    sampler="mh",
    sampler_kwargs={
        "proposal_scale": {
            "alpha": 0.15,
            "beta": 0.08,
            "sigma": 0.1,
        }
    },
    seed=43,
)

print(f"Elapsed time: {multi_result.elapsed_time:.2f}s")
for i, rate in enumerate(multi_result.acceptance_rate):
    print(f"Chain {i + 1}: acceptance rate = {rate:.1%}")

# Samples now have shape (num_chains, num_samples)
multi_chain_samples = multi_result.samples

In [ ]:
# Compute R-hat for each parameter
print("R-hat Diagnostics (values close to 1.0 indicate convergence):")
print("-" * 50)
for name in model.param_names:
    rhat_val = r_hat(multi_chain_samples[name])
    status = "OK" if rhat_val < 1.01 else "WARNING"
    print(f"{name}: R-hat = {rhat_val:.4f} [{status}]")

# The summary() method also includes r_hat automatically for multi-chain data
print("\n\nFull Summary (via InferenceResult.summary()):")
print("-" * 80)
print(f"{'Parameter':<10} {'Mean':>8} {'Std':>8} {'ESS':>8} {'R-hat':>8} {'5%':>8} {'95%':>8}")
print("-" * 80)

mc_stats = multi_result.summary()
for name, true_val in [("alpha", true_alpha), ("beta", true_beta), ("sigma", true_sigma)]:
    s = mc_stats[name]
    print(f"{name:<10} {s['mean']:>8.3f} {s['std']:>8.3f} {s['ess']:>8.1f} {s['r_hat']:>8.4f} {s['5%']:>8.3f} {s['95%']:>8.3f}")

In [ ]:
# Visualize chain mixing: trace plots with all chains overlaid
fig, axes = plt.subplots(3, 2, figsize=(14, 8))
num_chains = multi_result.num_chains
colors = plt.cm.tab10(np.linspace(0, 1, num_chains))

for i, (name, true_val) in enumerate([("alpha", true_alpha), ("beta", true_beta), ("sigma", true_sigma)]):
    # Trace plot with all chains
    for chain_idx in range(num_chains):
        axes[i, 0].plot(
            multi_chain_samples[name][chain_idx],
            alpha=0.7,
            lw=0.5,
            color=colors[chain_idx],
            label=f"Chain {chain_idx + 1}" if i == 0 else None,
        )
    axes[i, 0].axhline(true_val, color="red", linestyle="--", lw=1.5)
    axes[i, 0].set_ylabel(name)

    # Histogram: overlay all chains
    for chain_idx in range(num_chains):
        axes[i, 1].hist(
            multi_chain_samples[name][chain_idx],
            bins=30,
            density=True,
            alpha=0.4,
            color=colors[chain_idx],
        )
    axes[i, 1].axvline(true_val, color="red", linestyle="--", lw=2, label="True")
    axes[i, 1].set_xlabel(name)

axes[0, 0].set_title("Trace Plots (4 chains)")
axes[0, 1].set_title("Posterior Distributions")
axes[0, 0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print("\nGood chain mixing is indicated by:")
print("- Trace plots: Chains overlapping and exploring the same regions")
print("- Histograms: Similar distributions across chains")
print("- R-hat values close to 1.0")